<a href="https://colab.research.google.com/github/vg137/ibd-causal-multiomics/blob/main/nb3_integrating_gwas_and_expressions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction and setup

In this notebook, we will integrate the GWAS analysis and the gene set expression analysis. <font color="red">Add more details later!</font>

In [42]:
# For accessing data stored in the Google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [9]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns

In [49]:
data_dir = "/content/drive/MyDrive/Colab Notebooks/Genomics project/ibd_data"
snp2gene_subdir = "snp2gene"
snp2gene_filepath = f"{data_dir}/{snp2gene_subdir}/GCST90292538_snp2gene.parquet"

snp2gene_df = pd.read_parquet(snp2gene_filepath)
snp2gene_df.head()

,chromosome,base_pair_location,p_value,distance_to_assigned_gene,assigned_gene_id,assigned_gene_name,assigned_gene_type
0,1,1215852,2.250000e-09,1699.0,ENSG00000186827.12,TNFRSF4,protein_coding
1,1,1216593,8.490000e-10,2440.0,ENSG00000186827.12,TNFRSF4,protein_coding
2,1,1221049,7.534000e-10,6896.0,ENSG00000186827.12,TNFRSF4,protein_coding
3,1,1238902,1.230000e-08,32.0,ENSG00000078808.21,SDF4,protein_coding
4,1,1246371,6.517000e-09,351.0,ENSG00000184163.3,C1QTNF12,protein_coding


Let us again see how many unique significant gene symbols we have found in the GWAS data

In [53]:
len(set(snp2gene_df["assigned_gene_name"].values))

1643

Let us also load the expression data for the Hallmark gene sets

In [57]:
# Data subdirectories and files
gene_set_expr_subdir = "gene_set_expressions"
gene_set_expr_filepath = f"{data_dir}/{gene_set_expr_subdir}/hallmark_expressions.parquet"

hallmark_expr_df = pd.read_parquet(gene_set_expr_filepath)
hallmark_expr_df.head(3)

,GSM1598408,GSM1598409,GSM1598410,GSM1598411,GSM1598412,GSM1598413,GSM1598414,GSM1598415,GSM1598416,GSM1598417,...,GSM1598721,GSM1598722,GSM1598723,GSM1598724,GSM1598725,GSM1598726,GSM1598727,GSM1598728,GSM1598729,relevant_genes
gene_set_name,,,,,,,,,,,,,,,,,,,,,
HALLMARK_ADIPOGENESIS,4.673057,3.722162,4.276420,4.903253,3.979948,4.594571,4.655792,4.814399,4.477004,4.928155,...,4.824022,4.582675,4.766731,4.914146,4.799049,4.905908,5.022364,4.646561,4.765792,"[DNAJC15, GPX3, COL15A1, FABP4, MYLK, NDUFA5, ..."
HALLMARK_ALLOGRAFT_REJECTION,3.722815,4.230910,4.181253,3.623362,4.442373,4.238733,4.968203,3.896786,4.386900,3.478985,...,4.622982,4.185091,4.232684,4.835743,3.944240,4.614053,3.982420,4.118930,3.494830,"[ACVR2A, TAPBP, CCND2, CCL19, IKBKB, LY75, UBE..."
HALLMARK_ANDROGEN_RESPONSE,4.637928,3.634280,4.116811,4.644959,4.032413,4.509185,4.590260,4.483177,4.204952,4.578868,...,4.498439,4.492673,4.545977,4.716781,4.208033,4.490627,4.654086,4.345503,4.177016,"[HOMER2, CCND1, KLK2, TMPRSS2, B4GALT1, PGM3, ..."


Let us now find the genes which appear both in the gene sets in the Hallmark gene sets and the ones which were found in the GWAS analysis

In [143]:
gwas_genes = set(snp2gene_df["assigned_gene_name"])

expr_and_assoc_series = pd.Series(index=hallmark_expr_df.index,
                                    dtype=object,
                                 )

for gene_set_name, expressed_genes in hallmark_expr_df["relevant_genes"].items():
    common_genes = set(expressed_genes) & gwas_genes
    expr_and_assoc_series.loc[gene_set_name] = list(common_genes)

hallmark_intersection_df = (hallmark_expr_df[["relevant_genes"]]
                                .rename(columns={"relevant_genes": "expressed_genes"}))

hallmark_intersection_df["expr_and_assoc_genes"] = expr_and_assoc_series
hallmark_intersection_df["num_expressed_genes"] = hallmark_intersection_df["expressed_genes"].apply(len)
hallmark_intersection_df["num_expr_and_assoc_genes"] = expr_and_assoc_series.apply(len)
hallmark_intersection_df = hallmark_intersection_df.sort_values(by="num_expr_and_assoc_genes", ascending=False).head(5)
hallmark_intersection_df.head()

,expressed_genes,expr_and_assoc_genes,num_expressed_genes,num_expr_and_assoc_genes
gene_set_name,,,,
HALLMARK_ALLOGRAFT_REJECTION,"[ACVR2A, TAPBP, CCND2, CCL19, IKBKB, LY75, UBE...","[HLA-A, CCL11, IL13, CCR5, CCL2, IL18RAP, CCL7...",200,34
HALLMARK_INTERFERON_GAMMA_RESPONSE,"[NAMPT, DHX58, EIF4E3, IFI44L, C1R, OAS3, SRI,...","[HLA-A, CFB, DHX58, CCL2, CD274, CCL7, IRF5, C...",200,24
HALLMARK_INFLAMMATORY_RESPONSE,"[HPN, NAMPT, ACVR2A, CD55, BEST1, MYC, C5AR1, ...","[CSF3, CCL2, TNFRSF9, IL18RAP, CCL7, IFNGR2, I...",200,22
HALLMARK_IL2_STAT5_SIGNALING,"[CD83, MYC, CCND2, CAPN3, PLEC, S100A1, CD48, ...","[IL1RL1, AGER, GPX4, IL13, IL1R2, TNFRSF9, MAP...",199,19
HALLMARK_TNFA_SIGNALING_VIA_NFKB,"[NAMPT, PLPP3, CD83, MYC, SERPINB8, PDE4B, REL...","[CCL2, TNFRSF9, MAP3K8, IFNGR2, PLAU, IRF1, FO...",200,19


We have also ranked the gene sets based on the number of genes appearing in the intersection